### Guardrails
Guardrails enable you to do checks and validations of user input and agent output. 

[See the details](https://openai.github.io/openai-agents-python/guardrails/)

In [1]:
from agents import (Agent, 
                    Runner, AsyncOpenAI, 
                    OpenAIChatCompletionsModel, set_tracing_disabled)

In [2]:
# Tracing disabled
set_tracing_disabled(disabled=True)

In [3]:
import os
gemini_api = os.environ['GEMINI_API_KEY']

In [4]:
# 1. Which LLM Service?
external_client = AsyncOpenAI(
    api_key=gemini_api,
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
)

In [5]:
# 2. Which LLM Model?
llm_model = OpenAIChatCompletionsModel(
    model="gemini-2.5-flash",
    openai_client=external_client
)

##### Step-1

##### Input guardrails
1. First, the guardrail receives the same input passed to the agent.
2. Next, the guardrail function runs to produce a GuardrailFunctionOutput, which is then wrapped in an InputGuardrailResult.
3. Finally, we check if .tripwire_triggered is true. If true, an InputGuardrailTripwireTriggered exception is raised, so you can appropriately respond to the user or handle the exception.


In [ ]:
from pydantic import BaseModel
from agents import (
    GuardrailFunctionOutput,
    InputGuardrailTripwireTriggered,
    RunContextWrapper,
    TResponseInputItem,
)

from agents.decorators import input_guardrail

##### Step-2

In [ ]:
# Defining Schema
class MathHomeworkOutput(BaseModel):
    is_math_homework: bool
    reasoning: str

##### Step-3

In [8]:
# Defining the guardrail agent
guardrail_agent = Agent(  
    name="Guardrail check",
    instructions="Check if the user is asking you to do their math homework.",
    output_type=MathHomeworkOutput,
)

##### Step-4

In [9]:
@input_guardrail
async def math_guardrail(  
    ctx: RunContextWrapper[None], agent: Agent, input: str | list[TResponseInputItem]
) -> GuardrailFunctionOutput:
    result = await Runner.run(guardrail_agent, input, context=ctx.context)

    return GuardrailFunctionOutput(
        output_info=result.final_output,  
        tripwire_triggered=result.final_output.is_math_homework,
    )

##### Step-5

In [11]:
agent = Agent(  
    name="Customer support agent",
    instructions="You are a customer support agent. You help customers with their questions.",
    input_guardrails=[math_guardrail],
)

##### Step-6

In [12]:
try:
    await Runner.run(agent, "Hello, can you help me solve for x: 2x + 3 = 11?")
    print("Guardrail didn't trip - this is unexpected")

except InputGuardrailTripwireTriggered:
    print("Math homework guardrail tripped")

Math homework guardrail tripped


##### Step-7

In [13]:
try:
    await Runner.run(agent, "What is Machine Learning? Exlpain in two lines")
    print("Guardrail didn't trip - this is unexpected")

except InputGuardrailTripwireTriggered:
    print("Math homework guardrail tripped")

Guardrail didn't trip - this is unexpected
